# Phase 1 SFT — Qwen 3B + QLoRA + Weighted CE

Notebook này là bản đã sửa các lỗi thường gặp từ bản `phase1_sft.py`:

- Không dùng `group_by_length` để tránh lỗi version `TrainingArguments`.
- Không truyền cứng `tokenizer=` vào `Trainer`; tự kiểm tra version hỗ trợ `processing_class` hay `tokenizer`.
- `TrainingArguments` tự tương thích giữa `eval_strategy` và `evaluation_strategy`.
- Early stopping có ý nghĩa hơn: mặc định `NUM_EPOCHS = 3`.
- Dùng eval subset trong lúc train để đỡ chậm, nhưng vẫn full eval/test sau train.
- Test metric dùng `metric_key_prefix="test"`, không bị nhầm với `eval_loss`.
- Mount Drive một lần, zip adapter/checkpoint rồi copy sang Drive.
- Có cell debug active labels để kiểm tra loss có đang tính đúng trên `<analysis>` và `<final>` không.
- Metric năm được sửa để tránh ảo khi gold không có năm.

Mục tiêu Phase 1: train model sinh format:

```text
<analysis>
...
</analysis>
<final>
...
</final>
```

Khi deploy sau này, có thể chỉ hiển thị phần `<final>`.

In [1]:
# Cell 1 — Install dependencies
# Chạy cell này trước, sau đó nếu Colab yêu cầu thì restart runtime rồi chạy lại từ Cell 2.

%pip install -U \
  "transformers>=4.51.0" \
  "accelerate>=1.0.0" \
  "bitsandbytes>=0.43.0" \
  "peft>=0.13.0" \
  "datasets>=3.0.0" \
  "scikit-learn>=1.4.0" \
  "tensorboard" \
  "huggingface_hub"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 145.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 126.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 134.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 765.1/765.1 kB 56.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 54.7 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.6
    Uninstalling protobuf-5.29.6:
      Successfully uninstalled protobuf-5.29.6
  Attempting uninstall: tensorboard
    F

In [2]:
# Cell 2 — Imports

import os
import re
import json
import math
import random
import hashlib
import inspect
import shutil
from datetime import datetime
from dataclasses import dataclass
from typing import Any, Dict, List, Optional

import numpy as np
import torch
import torch.nn.functional as F

from datasets import load_dataset, DatasetDict
from sklearn.model_selection import GroupShuffleSplit

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    set_seed,
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)

In [3]:
# Cell 3 — Global config

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

# Dataset
DATASET_ID = "minhxthanh/Vietnam-History-1M-Vi"
DATASET_SPLIT = "train"

# Model 3B text instruct
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

# Output
OUTPUT_DIR = "./outputs/qwen2_5_3b_vnhistory_phase1_qlora"

# Sequence length
MAX_LENGTH = 1024
# MAX_LENGTH = 2048  # bật khi GPU đủ VRAM

# Start serious but safe. Để None nếu muốn dùng full 1M rows.
MAX_SAMPLES = 100_000
# MAX_SAMPLES = None

# Split
TRAIN_RATIO = 0.90
EVAL_RATIO = 0.05
TEST_RATIO = 0.05

# Weighted CE
REASONING_LOSS_WEIGHT = 0.5
FINAL_LOSS_WEIGHT = 1.0

# Training
# Early stopping chỉ thật sự hữu ích khi NUM_EPOCHS > 1.
NUM_EPOCHS = 1

# Effective batch = per_device_batch * gradient_accumulation * num_gpu
# Nếu OOM: giảm PER_DEVICE_* xuống 2 hoặc 1, tăng GRADIENT_ACCUMULATION_STEPS tương ứng.
PER_DEVICE_TRAIN_BATCH_SIZE = 4
PER_DEVICE_EVAL_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4

LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.03

LOGGING_STEPS = 20
EVAL_STEPS = 500
SAVE_STEPS = 500  # nên bằng EVAL_STEPS khi load_best_model_at_end=True

# Eval subset dùng trong lúc train để đỡ chậm. Full eval/test chạy sau train.
EVAL_DURING_TRAIN_SIZE = 1000

# Early stopping
EARLY_STOPPING_PATIENCE = 3
EARLY_STOPPING_THRESHOLD = 0.0

# LoRA
LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05

# Data processing
NUM_PROC = max(1, min(8, os.cpu_count() or 1))

# Google Drive backup
SAVE_ZIP_TO_DRIVE = True
DRIVE_SAVE_DIR = "/content/drive/MyDrive/vn_history_model_backups"

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Capability:", torch.cuda.get_device_capability(0))

print("Effective batch size per update:", PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS * max(1, torch.cuda.device_count()))

CUDA available: True
GPU: NVIDIA L4
Capability: (8, 9)
Effective batch size per update: 16


In [4]:
# Cell 4 — Optional: mount Google Drive once

if SAVE_ZIP_TO_DRIVE:
    try:
        from google.colab import drive
        if not os.path.exists("/content/drive/MyDrive"):
            drive.mount("/content/drive")
        os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
        print("Google Drive ready:", DRIVE_SAVE_DIR)
    except Exception as e:
        print("Could not mount Google Drive. You can ignore this outside Colab.")
        print(repr(e))

Mounted at /content/drive
Google Drive ready: /content/drive/MyDrive/vn_history_model_backups


In [5]:
# Cell 5 — Load dataset

raw_ds = load_dataset(DATASET_ID, split=DATASET_SPLIT)

print(raw_ds)
print(raw_ds.column_names)
print("First row:")
print(raw_ds[0])

if MAX_SAMPLES is not None and MAX_SAMPLES < len(raw_ds):
    raw_ds = raw_ds.shuffle(seed=SEED).select(range(MAX_SAMPLES))

print("Using dataset:", raw_ds)

README.md:   0%|          | 0.00/628 [00:00<?, ?B/s]

Vietnam History Complete.jsonl:   0%|          | 0.00/871M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000000 [00:00<?, ? examples/s]

Dataset({
    features: ['messages'],
    num_rows: 1000000
})
['messages']
First row:
{'messages': [{'role': 'system', 'content': 'Bạn là một trợ lý AI am hiểu về lịch sử Việt Nam. Trả lời bằng tiếng Việt, chính xác, súc tích.', 'channel': None}, {'role': 'user', 'content': 'Tóm tắt Hòa ước Patenôtre (Giáp Thân) và nêu ý nghĩa lịch sử của sự kiện này.', 'channel': None}, {'role': 'assistant', 'content': 'Hòa ước Patenôtre (Giáp Thân) (1884). Triều Nguyễn ký hiệp ước đặt Việt Nam dưới chế độ bảo hộ của Pháp. Chính thức hóa ách cai trị của Pháp trên toàn cõi Việt Nam.', 'channel': 'final'}]}
Using dataset: Dataset({
    features: ['messages'],
    num_rows: 100000
})


In [6]:
# Cell 6 — Parse messages helpers

def safe_parse_messages(x: Any) -> List[Dict[str, Any]]:
    """
    Robust parser for dataset messages.
    Expected common format:
    [
      {"role": "system", "content": "..."},
      {"role": "user", "content": "..."},
      {"role": "assistant", "channel": "analysis", "content": "..."},
      {"role": "assistant", "channel": "final", "content": "..."}
    ]
    """
    if x is None:
        return []

    if isinstance(x, list):
        return x

    if isinstance(x, str):
        try:
            y = json.loads(x)
            if isinstance(y, list):
                return y
            if isinstance(y, dict) and "messages" in y:
                return y["messages"]
        except Exception:
            return []

    return []


def normalize_text(s: Any) -> str:
    if s is None:
        return ""
    if not isinstance(s, str):
        s = str(s)
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    s = re.sub(r"\n{3,}", "\n\n", s)
    return s.strip()


def extract_parts(example: Dict[str, Any]) -> Dict[str, str]:
    """Extract system, user, analysis, final from one row."""
    messages = safe_parse_messages(example.get("messages"))

    system_parts = []
    user_parts = []
    analysis_parts = []
    final_parts = []
    other_assistant_parts = []

    for m in messages:
        if not isinstance(m, dict):
            continue

        role = normalize_text(m.get("role", "")).lower()
        channel = normalize_text(m.get("channel", "")).lower()
        content = normalize_text(m.get("content", ""))

        if not content:
            continue

        if role == "system":
            system_parts.append(content)
        elif role == "user":
            user_parts.append(content)
        elif role == "assistant":
            if channel == "analysis":
                analysis_parts.append(content)
            elif channel == "final":
                final_parts.append(content)
            else:
                other_assistant_parts.append(content)

    # fallback nếu dataset không có channel final rõ ràng
    if not final_parts and other_assistant_parts:
        final_parts = other_assistant_parts[-1:]

    # nếu assistant có 2 message nhưng không channel, message đầu có thể là analysis, cuối là final
    if not analysis_parts and not final_parts and len(other_assistant_parts) >= 2:
        analysis_parts = other_assistant_parts[:-1]
        final_parts = other_assistant_parts[-1:]

    return {
        "system": "\n\n".join(system_parts).strip(),
        "user": "\n\n".join(user_parts).strip(),
        "analysis": "\n\n".join(analysis_parts).strip(),
        "final": "\n\n".join(final_parts).strip(),
    }

sample_parts = extract_parts(raw_ds[0])
sample_parts

{'system': 'Bạn là một trợ lý AI am hiểu về lịch sử Việt Nam. Trả lời bằng tiếng Việt, chính xác, súc tích.',
 'user': 'Trình bày ngắn gọn về Lý Công Uẩn được tôn lập (kết thúc Tiền Lê) (1009).',
 'analysis': '',
 'final': '1009: Lý Công Uẩn được tôn lập (kết thúc Tiền Lê). Sau khi Lê Long Đĩnh qua đời, Lý Công Uẩn được tôn làm vua. Mở ra triều Lý với nhiều cải cách và thịnh trị.'}

In [7]:
# Cell 7 — Extract fields and filter valid rows

def add_parts_batch(batch):
    systems, users, analyses, finals, valid = [], [], [], [], []

    for i in range(len(batch["messages"])):
        ex = {k: batch[k][i] for k in batch.keys()}
        parts = extract_parts(ex)

        is_valid = bool(parts["user"]) and bool(parts["final"])

        systems.append(parts["system"])
        users.append(parts["user"])
        analyses.append(parts["analysis"])
        finals.append(parts["final"])
        valid.append(is_valid)

    return {
        "system_text": systems,
        "user_text": users,
        "analysis_text": analyses,
        "final_text": finals,
        "valid": valid,
    }


ds = raw_ds.map(
    add_parts_batch,
    batched=True,
    num_proc=NUM_PROC,
    desc="Extracting system/user/analysis/final",
)

before = len(ds)
ds = ds.filter(lambda x: x["valid"], num_proc=NUM_PROC)
after = len(ds)

print("Before:", before)
print("After:", after)
print("Dropped:", before - after)
print("Example after parse:")
print(ds[0])

Extracting system/user/analysis/final (num_proc=8):   0%|          | 0/100000 [00:00<?, ? examples/s]

Filter (num_proc=8):   0%|          | 0/100000 [00:00<?, ? examples/s]

Before: 100000
After: 100000
Dropped: 0
Example after parse:
{'messages': [{'role': 'system', 'content': 'Bạn là một trợ lý AI am hiểu về lịch sử Việt Nam. Trả lời bằng tiếng Việt, chính xác, súc tích.', 'channel': None}, {'role': 'user', 'content': 'Trình bày ngắn gọn về Lý Công Uẩn được tôn lập (kết thúc Tiền Lê) (1009).', 'channel': None}, {'role': 'assistant', 'content': '1009: Lý Công Uẩn được tôn lập (kết thúc Tiền Lê). Sau khi Lê Long Đĩnh qua đời, Lý Công Uẩn được tôn làm vua. Mở ra triều Lý với nhiều cải cách và thịnh trị.', 'channel': 'final'}], 'system_text': 'Bạn là một trợ lý AI am hiểu về lịch sử Việt Nam. Trả lời bằng tiếng Việt, chính xác, súc tích.', 'user_text': 'Trình bày ngắn gọn về Lý Công Uẩn được tôn lập (kết thúc Tiền Lê) (1009).', 'analysis_text': '', 'final_text': '1009: Lý Công Uẩn được tôn lập (kết thúc Tiền Lê). Sau khi Lê Long Đĩnh qua đời, Lý Công Uẩn được tôn làm vua. Mở ra triều Lý với nhiều cải cách và thịnh trị.', 'valid': True}


In [8]:
# Cell 8 — Create rough event_key for leakage-aware split
# Lưu ý: đây là heuristic cơ bản, không phải entity linker hoàn hảo.

VI_STOPWORDS = set("""
và của là có trong được các những một về với cho từ khi năm này đó đã để do tại
vì sao như thế nào ai gì nào hãy trình bày nêu giải thích phân tích cho biết
sự kiện nhân vật lịch sử việt nam thời kỳ triều đại cuộc khởi nghĩa chiến thắng
ý nghĩa nguyên nhân kết quả diễn biến vai trò bối cảnh ảnh hưởng
""".split())


def basic_normalize_for_key(text: str) -> str:
    text = text.lower()
    text = re.sub(r"[^\w\sÀ-ỹ]", " ", text, flags=re.UNICODE)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def extract_years(text: str) -> List[str]:
    years = re.findall(r"\b\d{3,4}\b", text or "")
    return years[:3]


def salient_tokens(text: str, max_tokens: int = 12) -> List[str]:
    text = basic_normalize_for_key(text or "")
    toks = text.split()

    out = []
    seen = set()
    for t in toks:
        if len(t) <= 2:
            continue
        if t in VI_STOPWORDS:
            continue
        if t.isdigit():
            continue
        if t not in seen:
            out.append(t)
            seen.add(t)
        if len(out) >= max_tokens:
            break
    return out


def make_event_key(user: str, final: str) -> str:
    text = f"{user}\n{final}"
    years = extract_years(text)
    toks = salient_tokens(text, max_tokens=12)

    if years or toks:
        raw_key = "years=" + ",".join(years) + "|tok=" + ",".join(toks)
    else:
        raw_key = basic_normalize_for_key(text)[:200]

    return hashlib.md5(raw_key.encode("utf-8")).hexdigest()


def add_event_key_batch(batch):
    return {
        "event_key": [make_event_key(u, f) for u, f in zip(batch["user_text"], batch["final_text"])]
    }


ds = ds.map(
    add_event_key_batch,
    batched=True,
    num_proc=NUM_PROC,
    desc="Creating event keys",
)

print("Example event_key:", ds[0]["event_key"])

Creating event keys (num_proc=8):   0%|          | 0/100000 [00:00<?, ? examples/s]

Example event_key: 0e118770e86f2c4f9702ebbfcc5064fc


In [9]:
# Cell 9 — Group split: train/eval/test

indices = np.arange(len(ds))
groups = np.array(ds["event_key"])

assert abs(TRAIN_RATIO + EVAL_RATIO + TEST_RATIO - 1.0) < 1e-6

# First split: train vs temp
gss1 = GroupShuffleSplit(
    n_splits=1,
    train_size=TRAIN_RATIO,
    random_state=SEED,
)
train_idx, temp_idx = next(gss1.split(indices, groups=groups))

temp_groups = groups[temp_idx]
temp_indices = indices[temp_idx]

# Second split: eval vs test
gss2 = GroupShuffleSplit(
    n_splits=1,
    train_size=EVAL_RATIO / (EVAL_RATIO + TEST_RATIO),
    random_state=SEED,
)
eval_rel_idx, test_rel_idx = next(gss2.split(temp_indices, groups=temp_groups))

eval_idx = temp_indices[eval_rel_idx]
test_idx = temp_indices[test_rel_idx]

dataset_raw = DatasetDict({
    "train": ds.select(train_idx.tolist()),
    "eval": ds.select(eval_idx.tolist()),
    "test": ds.select(test_idx.tolist()),
})

train_keys = set(dataset_raw["train"]["event_key"])
eval_keys = set(dataset_raw["eval"]["event_key"])
test_keys = set(dataset_raw["test"]["event_key"])

print(dataset_raw)
print("train ∩ eval:", len(train_keys & eval_keys))
print("train ∩ test:", len(train_keys & test_keys))
print("eval ∩ test:", len(eval_keys & test_keys))

# Exact question overlap check
train_users = set(dataset_raw["train"]["user_text"])
eval_users = set(dataset_raw["eval"]["user_text"])
test_users = set(dataset_raw["test"]["user_text"])
print("Exact user overlap train/eval:", len(train_users & eval_users))
print("Exact user overlap train/test:", len(train_users & test_users))
print("Exact user overlap eval/test:", len(eval_users & test_users))

DatasetDict({
    train: Dataset({
        features: ['messages', 'system_text', 'user_text', 'analysis_text', 'final_text', 'valid', 'event_key'],
        num_rows: 89810
    })
    eval: Dataset({
        features: ['messages', 'system_text', 'user_text', 'analysis_text', 'final_text', 'valid', 'event_key'],
        num_rows: 5289
    })
    test: Dataset({
        features: ['messages', 'system_text', 'user_text', 'analysis_text', 'final_text', 'valid', 'event_key'],
        num_rows: 4901
    })
})
train ∩ eval: 0
train ∩ test: 0
eval ∩ test: 0
Exact user overlap train/eval: 160
Exact user overlap train/test: 162
Exact user overlap eval/test: 17


In [10]:
# Cell 10 — Load tokenizer

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    use_fast=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

IM_START = "<|im_start|>"
IM_END = "<|im_end|>"

print("pad token:", tokenizer.pad_token, tokenizer.pad_token_id)
print("eos token:", tokenizer.eos_token, tokenizer.eos_token_id)
print("im_end id:", tokenizer.convert_tokens_to_ids(IM_END))

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

pad token: <|endoftext|> 151643
eos token: <|im_end|> 151645
im_end id: 151645


In [11]:
# Cell 11 — Tokenization with assistant-only weighted labels

DEFAULT_SYSTEM = (
    "Bạn là trợ lý AI chuyên về lịch sử Việt Nam. "
    "Hãy suy luận cẩn thận, trả lời chính xác, rõ ràng và không bịa thông tin."
)


def tok(text: str) -> List[int]:
    return tokenizer(text, add_special_tokens=False)["input_ids"]


def add_segment(
    input_ids: List[int],
    labels: List[int],
    loss_weights: List[float],
    text: str,
    train_on_segment: bool,
    weight: float,
):
    ids = tok(text)
    input_ids.extend(ids)

    if train_on_segment:
        labels.extend(ids)
        loss_weights.extend([float(weight)] * len(ids))
    else:
        labels.extend([-100] * len(ids))
        loss_weights.extend([0.0] * len(ids))


def build_training_example(
    system: str,
    user: str,
    analysis: str,
    final: str,
    max_length: int = MAX_LENGTH,
) -> Dict[str, Any]:
    system = normalize_text(system) or DEFAULT_SYSTEM
    user = normalize_text(user)
    analysis = normalize_text(analysis)
    final = normalize_text(final)

    input_ids = []
    labels = []
    loss_weights = []

    # System: masked
    add_segment(
        input_ids, labels, loss_weights,
        f"{IM_START}system\n{system}{IM_END}\n",
        train_on_segment=False,
        weight=0.0,
    )

    # User: masked
    add_segment(
        input_ids, labels, loss_weights,
        f"{IM_START}user\n{user}{IM_END}\n",
        train_on_segment=False,
        weight=0.0,
    )

    # Assistant header: masked
    add_segment(
        input_ids, labels, loss_weights,
        f"{IM_START}assistant\n",
        train_on_segment=False,
        weight=0.0,
    )

    # Assistant analysis: lower weight
    if analysis:
        add_segment(input_ids, labels, loss_weights, "<analysis>\n", True, REASONING_LOSS_WEIGHT)
        add_segment(input_ids, labels, loss_weights, analysis + "\n", True, REASONING_LOSS_WEIGHT)
        add_segment(input_ids, labels, loss_weights, "</analysis>\n", True, REASONING_LOSS_WEIGHT)

    # Assistant final: full weight
    add_segment(input_ids, labels, loss_weights, "<final>\n", True, FINAL_LOSS_WEIGHT)
    add_segment(input_ids, labels, loss_weights, final + "\n", True, FINAL_LOSS_WEIGHT)
    add_segment(input_ids, labels, loss_weights, f"</final>{IM_END}\n", True, FINAL_LOSS_WEIGHT)

    attention_mask = [1] * len(input_ids)
    too_long = len(input_ids) > max_length
    has_loss = any(x != -100 for x in labels)

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
        "loss_weights": loss_weights,
        "length": len(input_ids),
        "too_long": too_long,
        "has_loss": has_loss,
    }


def tokenize_batch(batch):
    out = {
        "input_ids": [],
        "attention_mask": [],
        "labels": [],
        "loss_weights": [],
        "length": [],
        "too_long": [],
        "has_loss": [],
    }

    for system, user, analysis, final in zip(
        batch["system_text"],
        batch["user_text"],
        batch["analysis_text"],
        batch["final_text"],
    ):
        item = build_training_example(system, user, analysis, final, max_length=MAX_LENGTH)
        for k in out:
            out[k].append(item[k])

    return out

In [12]:
# Cell 12 — Tokenize dataset and filter too-long examples

remove_cols = dataset_raw["train"].column_names

tokenized = dataset_raw.map(
    tokenize_batch,
    batched=True,
    num_proc=NUM_PROC,
    remove_columns=remove_cols,
    desc="Tokenizing with weighted labels",
)

before_train = len(tokenized["train"])
before_eval = len(tokenized["eval"])
before_test = len(tokenized["test"])

tokenized = tokenized.filter(
    lambda x: (not x["too_long"]) and x["has_loss"],
    num_proc=NUM_PROC,
    desc="Filtering too-long/invalid examples",
)

print("Train before/after:", before_train, len(tokenized["train"]))
print("Eval before/after:", before_eval, len(tokenized["eval"]))
print("Test before/after:", before_test, len(tokenized["test"]))

lengths = tokenized["train"]["length"]
print("Length mean:", np.mean(lengths))
print("Length p50:", np.percentile(lengths, 50))
print("Length p90:", np.percentile(lengths, 90))
print("Length p95:", np.percentile(lengths, 95))
print("Length p99:", np.percentile(lengths, 99))
print("Length max:", np.max(lengths))

# Eval subset trong lúc train để tiết kiệm thời gian.
eval_during_train = tokenized["eval"]
if len(eval_during_train) > EVAL_DURING_TRAIN_SIZE:
    eval_during_train = eval_during_train.shuffle(seed=SEED).select(range(EVAL_DURING_TRAIN_SIZE))
print("Eval during train size:", len(eval_during_train))

Tokenizing with weighted labels (num_proc=8):   0%|          | 0/89810 [00:00<?, ? examples/s]

Tokenizing with weighted labels (num_proc=8):   0%|          | 0/5289 [00:00<?, ? examples/s]

Tokenizing with weighted labels (num_proc=8):   0%|          | 0/4901 [00:00<?, ? examples/s]

Filtering too-long/invalid examples (num_proc=8):   0%|          | 0/89810 [00:00<?, ? examples/s]

Filtering too-long/invalid examples (num_proc=8):   0%|          | 0/5289 [00:00<?, ? examples/s]

Filtering too-long/invalid examples (num_proc=8):   0%|          | 0/4901 [00:00<?, ? examples/s]

Train before/after: 89810 89810
Eval before/after: 5289 5289
Test before/after: 4901 4901
Length mean: 192.5975503841443
Length p50: 203.0
Length p90: 240.0
Length p95: 254.0
Length p99: 270.0
Length max: 292
Eval during train size: 1000


In [13]:
# Cell 13 — Data collator for loss_weights

@dataclass
class WeightedDataCollator:
    tokenizer: Any
    pad_to_multiple_of: Optional[int] = 8

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        max_len = max(len(f["input_ids"]) for f in features)

        if self.pad_to_multiple_of is not None:
            if max_len % self.pad_to_multiple_of != 0:
                max_len = ((max_len // self.pad_to_multiple_of) + 1) * self.pad_to_multiple_of

        batch_input_ids = []
        batch_attention_mask = []
        batch_labels = []
        batch_loss_weights = []

        pad_id = self.tokenizer.pad_token_id

        for f in features:
            input_ids = f["input_ids"]
            attention_mask = f["attention_mask"]
            labels = f["labels"]
            loss_weights = f["loss_weights"]

            pad_len = max_len - len(input_ids)

            batch_input_ids.append(input_ids + [pad_id] * pad_len)
            batch_attention_mask.append(attention_mask + [0] * pad_len)
            batch_labels.append(labels + [-100] * pad_len)
            batch_loss_weights.append(loss_weights + [0.0] * pad_len)

        return {
            "input_ids": torch.tensor(batch_input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(batch_attention_mask, dtype=torch.long),
            "labels": torch.tensor(batch_labels, dtype=torch.long),
            "loss_weights": torch.tensor(batch_loss_weights, dtype=torch.float),
        }


data_collator = WeightedDataCollator(tokenizer=tokenizer)

In [14]:
# Cell 14 — Load Qwen 3B with 4-bit quantization

major_cc = torch.cuda.get_device_capability(0)[0] if torch.cuda.is_available() else 0
USE_BF16 = torch.cuda.is_available() and major_cc >= 8
compute_dtype = torch.bfloat16 if USE_BF16 else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=compute_dtype,
)

model.config.use_cache = False

print("Loaded model:", MODEL_ID)
print("Compute dtype:", compute_dtype)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded model: Qwen/Qwen2.5-3B-Instruct
Compute dtype: torch.bfloat16


In [15]:
# Cell 15 — Prepare QLoRA adapter

model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 59,867,136 || all params: 3,145,805,824 || trainable%: 1.9031


In [16]:
# Cell 16 — Custom Trainer with weighted cross entropy

class WeightedCETrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        loss_weights = inputs.pop("loss_weights")

        outputs = model(**inputs)
        logits = outputs.logits
        labels = inputs["labels"]

        # Shift for causal LM:
        # logits[:, t] predicts labels[:, t+1]
        shift_logits = logits[:, :-1, :].contiguous()
        shift_labels = labels[:, 1:].contiguous()
        shift_weights = loss_weights[:, 1:].contiguous()

        vocab_size = shift_logits.size(-1)

        flat_logits = shift_logits.view(-1, vocab_size)
        flat_labels = shift_labels.view(-1)
        flat_weights = shift_weights.view(-1)

        token_loss = F.cross_entropy(
            flat_logits,
            flat_labels,
            reduction="none",
            ignore_index=-100,
        )

        active = flat_labels.ne(-100)
        active_weights = flat_weights[active]
        active_loss = token_loss[active]

        loss = (active_loss * active_weights).sum() / active_weights.sum().clamp(min=1.0)

        return (loss, outputs) if return_outputs else loss

In [17]:
# Cell 17 — Version-compatible TrainingArguments
# Fix:
# - Không dùng group_by_length
# - Tự xử lý eval_strategy/evaluation_strategy
# - Ép save_steps = eval_steps để load_best_model_at_end hoạt động đúng
# - Giữ best checkpoint theo eval_loss
# - Tương thích nhiều version transformers

import inspect
from transformers import TrainingArguments

def build_training_arguments() -> TrainingArguments:
    params = inspect.signature(TrainingArguments.__init__).parameters

    # Rất quan trọng:
    # Nếu load_best_model_at_end=True thì save/eval nên cùng nhịp.
    # Nếu không, best eval checkpoint có thể không tồn tại để load.
    effective_eval_steps = int(EVAL_STEPS)
    effective_save_steps = int(EVAL_STEPS)

    if SAVE_STEPS != EVAL_STEPS:
        print(
            f"Warning: SAVE_STEPS={SAVE_STEPS} khác EVAL_STEPS={EVAL_STEPS}. "
            f"Sẽ dùng save_steps={effective_save_steps} để khớp với eval_steps."
        )

    kwargs = dict(
        output_dir=OUTPUT_DIR,

        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,

        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        warmup_ratio=WARMUP_RATIO,
        lr_scheduler_type="cosine",

        logging_steps=LOGGING_STEPS,

        # Save theo steps, cùng nhịp với eval.
        save_strategy="steps",
        save_steps=effective_save_steps,
        save_total_limit=3,

        # Best model.
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,

        bf16=USE_BF16,
        fp16=not USE_BF16,

        optim="paged_adamw_8bit",

        gradient_checkpointing=True,
        max_grad_norm=1.0,

        report_to=["tensorboard"],

        remove_unused_columns=False,
        dataloader_num_workers=2,
        dataloader_pin_memory=True,

        seed=SEED,
    )

    # Evaluation strategy khác tên giữa các version transformers.
    if "eval_strategy" in params:
        kwargs["eval_strategy"] = "steps"
    elif "evaluation_strategy" in params:
        kwargs["evaluation_strategy"] = "steps"
    else:
        print(
            "Warning: version transformers này không có eval_strategy/evaluation_strategy. "
            "Sẽ tắt load_best_model_at_end vì không thể eval trong lúc train."
        )
        kwargs["load_best_model_at_end"] = False
        kwargs.pop("metric_for_best_model", None)
        kwargs.pop("greater_is_better", None)

    if "eval_steps" in params:
        kwargs["eval_steps"] = effective_eval_steps
    else:
        print(
            "Warning: version transformers này không hỗ trợ eval_steps. "
            "Early stopping theo steps có thể không hoạt động đúng."
        )

    # Một số version không nhận key này.
    if "do_eval" in params:
        kwargs["do_eval"] = True

    # Filter unsupported keys.
    filtered = {k: v for k, v in kwargs.items() if k in params}
    dropped = sorted(set(kwargs) - set(filtered))

    if dropped:
        print("Dropped unsupported TrainingArguments keys:", dropped)

    args = TrainingArguments(**filtered)

    print("TrainingArguments created.")
    print("NUM_EPOCHS:", NUM_EPOCHS)
    print("EVAL_STEPS:", effective_eval_steps)
    print("SAVE_STEPS actually used:", effective_save_steps)
    print("load_best_model_at_end:", getattr(args, "load_best_model_at_end", None))
    print("metric_for_best_model:", getattr(args, "metric_for_best_model", None))
    print("greater_is_better:", getattr(args, "greater_is_better", None))

    return args


training_args = build_training_arguments()
print(training_args)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


TrainingArguments created.
NUM_EPOCHS: 1
EVAL_STEPS: 500
SAVE_STEPS actually used: 500
load_best_model_at_end: True
metric_for_best_model: eval_loss
greater_is_better: False
TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=True,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=2,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_static_graph=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_ji

In [18]:
# Cell 18 — Create trainer with version-compatible tokenizer/processing_class and early stopping

def build_trainer() -> WeightedCETrainer:
    trainer_kwargs = dict(
        model=model,
        args=training_args,
        train_dataset=tokenized["train"],
        eval_dataset=eval_during_train,
        data_collator=data_collator,
        callbacks=[
            EarlyStoppingCallback(
                early_stopping_patience=EARLY_STOPPING_PATIENCE,
                early_stopping_threshold=EARLY_STOPPING_THRESHOLD,
            )
        ],
    )

    params = inspect.signature(WeightedCETrainer.__init__).parameters

    # Newer transformers: processing_class. Older: tokenizer. Some local versions: neither.
    if "processing_class" in params:
        trainer_kwargs["processing_class"] = tokenizer
    elif "tokenizer" in params:
        trainer_kwargs["tokenizer"] = tokenizer

    return WeightedCETrainer(**trainer_kwargs)


trainer = build_trainer()
print("Trainer created successfully.")

Trainer created successfully.


In [19]:
# Cell 19 — Sanity check one batch and debug active labels
# Quan trọng nếu loss quá thấp: cell này xác nhận loss đang tính trên analysis/final, không phải vài token rác.

batch = next(iter(trainer.get_train_dataloader()))

for k, v in batch.items():
    print(k, v.shape, v.dtype, v.device if hasattr(v, "device") else "")

with torch.no_grad():
    batch_on_device = {k: v.to(model.device) for k, v in batch.items()}
    sanity_loss = trainer.compute_loss(model, batch_on_device)

print("Sanity loss:", sanity_loss.item())

labels = batch["labels"]
loss_weights = batch["loss_weights"]
input_ids = batch["input_ids"]

active = labels.ne(-100)
weighted_active = loss_weights.gt(0)

print("input shape:", input_ids.shape)
print("active label tokens:", active.sum().item())
print("weighted active tokens:", weighted_active.sum().item())
print("active ratio:", active.float().mean().item())
print("weighted active ratio:", weighted_active.float().mean().item())

for i in range(min(3, input_ids.shape[0])):
    print("=" * 100)
    active_ids = input_ids[i][labels[i].ne(-100)]
    print(tokenizer.decode(active_ids[:700], skip_special_tokens=False))

input_ids torch.Size([4, 216]) torch.int64 cuda:0
attention_mask torch.Size([4, 216]) torch.int64 cuda:0
labels torch.Size([4, 216]) torch.int64 cuda:0
loss_weights torch.Size([4, 216]) torch.float32 cuda:0
Sanity loss: 2.501093626022339
input shape: torch.Size([4, 216])
active label tokens: 394
weighted active tokens: 394
active ratio: 0.45601850748062134
weighted active ratio: 0.45601850748062134
<final>
Lê Lợi lên ngôi, lập nhà Hậu Lê (1428). Lê Lợi xưng vương, ban Bình Ngô đại cáo tuyên cáo độc lập. Mở ra thời kỳ Lê sơ thịnh trị.
</final><|im_end|>

<analysis>
B1: gắn mốc 1054 với Đổi quốc hiệu thành Đại Việt. B2: nêu diễn biến trọng tâm – "Thời Lý Thánh Tông, quốc hiệu đổi từ Đại Cồ Việt sang Đại Việt.". B3: kết luận ý nghĩa – Khẳng định bản sắc và vị thế quốc gia độc lập..
</analysis>
<final>
**Đổi quốc hiệu thành Đại Việt** xảy ra năm 1054. Thời Lý Thánh Tông, quốc hiệu đổi từ Đại Cồ Việt sang Đại Việt. Đây là cột mốc quan trọng vì Khẳng định bản sắc và vị thế quốc gia độc lập.


In [20]:
# Cell 20 — Train

train_result = trainer.train()
train_result

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss
500,0.108247,0.028591
1000,0.105716,0.027051
1500,0.101466,0.024899
2000,0.106327,0.025941
2500,0.099668,0.024971
3000,0.098045,0.024750
3500,0.097196,0.024575
4000,0.096175,0.024227
4500,0.097576,0.024539
5000,0.097385,0.024215


TrainOutput(global_step=5614, training_loss=0.15442278954665942, metrics={'train_runtime': 26129.9098, 'train_samples_per_second': 3.437, 'train_steps_per_second': 0.215, 'total_flos': 3.577451966429921e+17, 'train_loss': 0.15442278954665942, 'epoch': 1.0})

In [21]:
# Cell 21 — Save adapter + tokenizer locally

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Saved adapter/tokenizer to:", OUTPUT_DIR)

Saved adapter/tokenizer to: ./outputs/qwen2_5_3b_vnhistory_phase1_qlora


In [22]:
# Cell 22 — Full eval and test metrics
# Chạy sau trainer.train().
# Nếu load_best_model_at_end=True, lúc này trainer.model đã là best checkpoint theo eval_loss.

import math
import json
import os

def safe_perplexity(loss_value):
    """Tính perplexity = exp(loss), tránh overflow."""
    if loss_value is None:
        return None
    try:
        if loss_value > 50:
            return float("inf")
        return math.exp(loss_value)
    except Exception:
        return None


print("=" * 100)
print("Best checkpoint selected during training:")
print("best_model_checkpoint:", trainer.state.best_model_checkpoint)
print("best_metric:", trainer.state.best_metric)
print("=" * 100)


# 1. Full validation eval
eval_metrics = trainer.evaluate(
    eval_dataset=tokenized["eval"],
    metric_key_prefix="eval",
)

eval_loss = eval_metrics.get("eval_loss")
eval_ppl = safe_perplexity(eval_loss)

print("\nFULL EVAL METRICS")
print(eval_metrics)

if eval_loss is not None:
    print("Eval loss:", eval_loss)
    print("Eval perplexity approx:", eval_ppl)


# 2. Full test eval
test_metrics = trainer.evaluate(
    eval_dataset=tokenized["test"],
    metric_key_prefix="test",
)

test_loss = test_metrics.get("test_loss")
test_ppl = safe_perplexity(test_loss)

print("\nFULL TEST METRICS")
print(test_metrics)

if test_loss is not None:
    print("Test loss:", test_loss)
    print("Test perplexity approx:", test_ppl)


# 3. Gom metrics lại để lưu
all_metrics = {
    "best_model_checkpoint": trainer.state.best_model_checkpoint,
    "best_metric": trainer.state.best_metric,

    "eval_loss": eval_loss,
    "eval_perplexity": eval_ppl,

    "test_loss": test_loss,
    "test_perplexity": test_ppl,

    "raw_eval_metrics": eval_metrics,
    "raw_test_metrics": test_metrics,
}

metrics_path = os.path.join(OUTPUT_DIR, "full_eval_test_metrics.json")

os.makedirs(OUTPUT_DIR, exist_ok=True)

with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(all_metrics, f, ensure_ascii=False, indent=2)

print("\nSaved metrics to:", metrics_path)

Best checkpoint selected during training:
best_model_checkpoint: ./outputs/qwen2_5_3b_vnhistory_phase1_qlora/checkpoint-5000
best_metric: 0.024214614182710648


Training Loss,Validation Loss,Step
0.095369,0.024173,5614



FULL EVAL METRICS
{'eval_loss': 0.024172600358724594}
Eval loss: 0.024172600358724594
Eval perplexity approx: 1.0244671260250624


[transformers] early stopping required metric_for_best_model, but did not find eval_loss so early stopping is disabled


Training Loss,Validation Loss,Step
0.095369,0.024198,5614



FULL TEST METRICS
{'test_loss': 0.024198250845074654}
Test loss: 0.024198250845074654
Test perplexity approx: 1.0244934044421201

Saved metrics to: ./outputs/qwen2_5_3b_vnhistory_phase1_qlora/full_eval_test_metrics.json


In [23]:
# Cell 23 — Zip output folder and copy to Google Drive

if SAVE_ZIP_TO_DRIVE:
    try:
        from google.colab import drive
        if not os.path.exists("/content/drive/MyDrive"):
            drive.mount("/content/drive")
        os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        zip_base_name = f"qwen_vnhistory_phase1_adapter_{timestamp}"

        local_zip_base = f"/content/{zip_base_name}"
        local_zip_path = f"{local_zip_base}.zip"
        drive_zip_path = os.path.join(DRIVE_SAVE_DIR, f"{zip_base_name}.zip")

        if not os.path.exists(OUTPUT_DIR):
            raise FileNotFoundError(f"OUTPUT_DIR không tồn tại: {OUTPUT_DIR}")

        shutil.make_archive(
            base_name=local_zip_base,
            format="zip",
            root_dir=OUTPUT_DIR,
        )

        print("Created local zip:", local_zip_path)
        print("Zip size MB:", os.path.getsize(local_zip_path) / 1024 / 1024)

        shutil.copy2(local_zip_path, drive_zip_path)

        print("Copied to Google Drive:")
        print(drive_zip_path)
        print("Drive zip size MB:", os.path.getsize(drive_zip_path) / 1024 / 1024)

    except Exception as e:
        print("Drive backup failed. Local save still exists at:", OUTPUT_DIR)
        print(repr(e))
else:
    print("SAVE_ZIP_TO_DRIVE=False, skipped Drive backup.")

Created local zip: /content/qwen_vnhistory_phase1_adapter_20260704_181043.zip
Zip size MB: 1162.1947050094604
Copied to Google Drive:
/content/drive/MyDrive/vn_history_model_backups/qwen_vnhistory_phase1_adapter_20260704_181043.zip
Drive zip size MB: 1162.1947050094604


In [28]:
# Cell — Export clean best adapter only and copy zip to Google Drive
# Chạy sau trainer.train().
# Nếu load_best_model_at_end=True, trainer.model hiện tại đã là best checkpoint theo eval_loss.

from google.colab import drive
from pathlib import Path
import os
import shutil
from datetime import datetime

# 1. Mount Google Drive nếu chưa mount
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")

# 2. In thông tin best checkpoint
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best metric:", trainer.state.best_metric)

if trainer.state.best_model_checkpoint is None:
    print(
        "Warning: trainer.state.best_model_checkpoint is None. "
        "Sẽ save model hiện tại. Kiểm tra lại load_best_model_at_end=True nếu cần."
    )

# 3. Tạo folder export sạch
CLEAN_EXPORT_DIR = "/content/qwen_vnhistory_phase1_best_adapter_clean"

if os.path.exists(CLEAN_EXPORT_DIR):
    shutil.rmtree(CLEAN_EXPORT_DIR)

os.makedirs(CLEAN_EXPORT_DIR, exist_ok=True)

# 4. Save adapter sạch
# Quan trọng:
# - Không copy cả OUTPUT_DIR
# - Không copy checkpoint-xxxx
# - Chỉ save adapter/tokenizer hiện tại
# Nếu load_best_model_at_end=True, đây là best adapter.
trainer.save_model(CLEAN_EXPORT_DIR)
tokenizer.save_pretrained(CLEAN_EXPORT_DIR)

# 5. Copy metrics nếu có
metrics_files = [
    "full_eval_test_metrics.json",
    "generation_year_metrics.json",
    "generation_year_examples.json",
    "trainer_state.json",
]

for fname in metrics_files:
    src = os.path.join(OUTPUT_DIR, fname)
    dst = os.path.join(CLEAN_EXPORT_DIR, fname)
    if os.path.exists(src):
        shutil.copy2(src, dst)

# 6. Helper tính dung lượng
def get_size_mb(path):
    path = Path(path)
    total = 0

    if path.is_file():
        return path.stat().st_size / 1024 / 1024

    for p in path.rglob("*"):
        if p.is_file():
            total += p.stat().st_size

    return total / 1024 / 1024

print("\nClean export folder:", CLEAN_EXPORT_DIR)
print("Clean export size MB:", get_size_mb(CLEAN_EXPORT_DIR))

print("\nFiles in clean export:")
for p in sorted(Path(CLEAN_EXPORT_DIR).iterdir()):
    print(f"{p.name:40s} {get_size_mb(p):10.2f} MB")

# 7. Zip clean adapter
DRIVE_SAVE_DIR = "/content/drive/MyDrive/vn_history_model_backups"
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
zip_base_name = f"qwen_vnhistory_phase1_best_adapter_clean_{timestamp}"

local_zip_base = f"/content/{zip_base_name}"
local_zip_path = f"{local_zip_base}.zip"

drive_zip_path = os.path.join(DRIVE_SAVE_DIR, f"{zip_base_name}.zip")

# Xóa zip cũ cùng tên nếu có
if os.path.exists(local_zip_path):
    os.remove(local_zip_path)

shutil.make_archive(
    base_name=local_zip_base,
    format="zip",
    root_dir=CLEAN_EXPORT_DIR,
)

print("\nCreated local clean zip:", local_zip_path)
print("Local zip size MB:", os.path.getsize(local_zip_path) / 1024 / 1024)

# 8. Copy zip sang Google Drive
shutil.copy2(local_zip_path, drive_zip_path)

print("\nCopied clean best adapter zip to Google Drive:")
print(drive_zip_path)
print("Drive zip size MB:", os.path.getsize(drive_zip_path) / 1024 / 1024)

Best checkpoint: ./outputs/qwen2_5_3b_vnhistory_phase1_qlora/checkpoint-5000
Best metric: 0.024214614182710648

Clean export folder: /content/qwen_vnhistory_phase1_best_adapter_clean
Clean export size MB: 239.34703636169434

Files in clean export:
README.md                                      0.00 MB
adapter_config.json                            0.00 MB
adapter_model.safetensors                    228.44 MB
chat_template.jinja                            0.00 MB
full_eval_test_metrics.json                    0.00 MB
tokenizer.json                                10.89 MB
tokenizer_config.json                          0.00 MB
training_args.bin                              0.00 MB

Created local clean zip: /content/qwen_vnhistory_phase1_best_adapter_clean_20260704_183413.zip
Local zip size MB: 213.7997579574585

Copied clean best adapter zip to Google Drive:
/content/drive/MyDrive/vn_history_model_backups/qwen_vnhistory_phase1_best_adapter_clean_20260704_183413.zip
Drive zip size MB: 213

In [24]:
# Cell 24 — Inference helper

@torch.inference_mode()
def generate_answer(
    question: str,
    system: Optional[str] = None,
    max_new_tokens: int = 512,
    temperature: float = 0.2,
    top_p: float = 0.9,
):
    model.eval()

    system = normalize_text(system) or DEFAULT_SYSTEM
    question = normalize_text(question)

    prompt = (
        f"{IM_START}system\n{system}{IM_END}\n"
        f"{IM_START}user\n{question}{IM_END}\n"
        f"{IM_START}assistant\n"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    im_end_id = tokenizer.convert_tokens_to_ids(IM_END)
    eos_ids = [tokenizer.eos_token_id]
    if isinstance(im_end_id, int) and im_end_id >= 0:
        eos_ids.append(im_end_id)

    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=temperature > 0,
        temperature=temperature,
        top_p=top_p,
        eos_token_id=eos_ids,
        pad_token_id=tokenizer.pad_token_id,
    )

    generated = output_ids[0][inputs["input_ids"].shape[-1]:]
    text = tokenizer.decode(generated, skip_special_tokens=False)

    return text


def extract_final(text: str) -> str:
    if "<final>" in text:
        part = text.split("<final>", 1)[1]
        if "</final>" in part:
            part = part.split("</final>", 1)[0]
        return part.strip()

    cleaned = text.replace(IM_END, "")
    if tokenizer.eos_token:
        cleaned = cleaned.replace(tokenizer.eos_token, "")
    return cleaned.strip()

In [25]:
# Cell 25 — Quick manual tests

questions = [
    "Vì sao chiến thắng Bạch Đằng năm 938 được xem là bước ngoặt lớn trong lịch sử Việt Nam?",
    "Hãy giải thích ngắn gọn ý nghĩa của việc Lý Công Uẩn dời đô ra Thăng Long.",
    "So sánh vai trò của nhà Lý và nhà Trần trong việc xây dựng quốc gia Đại Việt.",
    "Khởi nghĩa Lam Sơn diễn ra trong bối cảnh nào và kết quả ra sao?",
]

for q in questions:
    raw = generate_answer(q, max_new_tokens=512, temperature=0.2)
    final = extract_final(raw)

    print("=" * 100)
    print("QUESTION:", q)
    print("\nRAW OUTPUT:\n", raw)
    print("\nFINAL ONLY:\n", final)

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


QUESTION: Vì sao chiến thắng Bạch Đằng năm 938 được xem là bước ngoặt lớn trong lịch sử Việt Nam?

RAW OUTPUT:
 <analysis>
B1: gắn mốc 938 với chiến thắng Bạch Đằng. B2: nêu diễn biến trọng tâm – "Ngô Quyền dùng cọc gỗ đặt ngầm trên sông Bạch Đằng, đánh bại thủy quân Nam Hán.". B3: kết luận ý nghĩa – Chấm dứt hơn một thiên niên kỷ Bắc thuộc, mở ra kỷ nguyên độc lập của người Việt..
</analysis>
<final>
**Chiến thắng Bạch Đằng** xảy ra năm 938. Ngô Quyền dùng cọc gỗ đặt ngầm trên sông Bạch Đằng, đánh bại thủy quân Nam Hán. Đây là cột mốc quan trọng vì Chấm dứt hơn một thiên niên kỷ Bắc thuộc, mở ra kỷ nguyên độc lập của người Việt.
</final><|im_end|>

FINAL ONLY:
 **Chiến thắng Bạch Đằng** xảy ra năm 938. Ngô Quyền dùng cọc gỗ đặt ngầm trên sông Bạch Đằng, đánh bại thủy quân Nam Hán. Đây là cột mốc quan trọng vì Chấm dứt hơn một thiên niên kỷ Bắc thuộc, mở ra kỷ nguyên độc lập của người Việt.
QUESTION: Hãy giải thích ngắn gọn ý nghĩa của việc Lý Công Uẩn dời đô ra Thăng Long.

RAW OUTPUT

In [27]:
# Cell 26 — Faster batched year metric
# Chỉ evaluate trên examples whose gold final contains at least one year.
# Nhanh hơn bản generate từng câu một.

import re
import json
import random
import numpy as np
from tqdm.auto import tqdm
from typing import Dict, List

def extract_year_set(text: str) -> set:
    return set(re.findall(r"\b\d{3,4}\b", text or ""))


def year_prf(pred: str, gold: str) -> Dict[str, float]:
    p = extract_year_set(pred)
    g = extract_year_set(gold)

    if not g:
        return {"year_precision": np.nan, "year_recall": np.nan, "year_f1": np.nan}

    if not p:
        return {"year_precision": 0.0, "year_recall": 0.0, "year_f1": 0.0}

    inter = len(p & g)
    precision = inter / max(1, len(p))
    recall = inter / max(1, len(g))
    f1 = 2 * precision * recall / max(1e-9, precision + recall)

    return {
        "year_precision": precision,
        "year_recall": recall,
        "year_f1": f1,
    }


def build_prompt(question: str, system: str = None) -> str:
    system = normalize_text(system) or DEFAULT_SYSTEM
    question = normalize_text(question)

    return (
        f"{IM_START}system\n{system}{IM_END}\n"
        f"{IM_START}user\n{question}{IM_END}\n"
        f"{IM_START}assistant\n"
    )


def extract_final(text: str) -> str:
    if "<final>" in text:
        part = text.split("<final>", 1)[1]
        if "</final>" in part:
            part = part.split("</final>", 1)[0]
        return part.strip()

    cleaned = text.replace(IM_END, "")
    if tokenizer.eos_token:
        cleaned = cleaned.replace(tokenizer.eos_token, "")
    return cleaned.strip()


@torch.inference_mode()
def batch_generate_answers(
    questions: List[str],
    max_new_tokens: int = 256,
    temperature: float = 0.0,
    top_p: float = 1.0,
) -> List[str]:
    model.eval()

    prompts = [build_prompt(q) for q in questions]

    # Với decoder-only LM, batch generation nên dùng left padding.
    old_padding_side = tokenizer.padding_side
    tokenizer.padding_side = "left"

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
    ).to(model.device)

    tokenizer.padding_side = old_padding_side

    im_end_id = tokenizer.convert_tokens_to_ids(IM_END)
    eos_ids = [tokenizer.eos_token_id]
    if isinstance(im_end_id, int) and im_end_id >= 0:
        eos_ids.append(im_end_id)

    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=temperature > 0,
        temperature=temperature if temperature > 0 else None,
        top_p=top_p if temperature > 0 else None,
        eos_token_id=eos_ids,
        pad_token_id=tokenizer.pad_token_id,
        use_cache=True,
    )

    generated_texts = []

    for i in range(len(questions)):
        prompt_len = inputs["input_ids"].shape[1]
        gen_ids = output_ids[i][prompt_len:]
        text = tokenizer.decode(gen_ids, skip_special_tokens=False)
        generated_texts.append(text)

    return generated_texts


@torch.inference_mode()
def evaluate_year_metric_only_gold_has_year_fast(
    raw_eval_ds,
    n: int = 50,
    batch_size: int = 4,
    max_new_tokens: int = 256,
):
    # 1. Lọc những mẫu gold có năm
    candidates = [
        ex for ex in raw_eval_ds
        if len(extract_year_set(ex["final_text"])) > 0
    ]

    print("Candidates with year:", len(candidates))

    random.seed(SEED)
    random.shuffle(candidates)
    sample = candidates[:min(n, len(candidates))]

    scores = []
    examples = []

    # 2. Bật cache cho generation
    old_use_cache = getattr(model.config, "use_cache", None)
    model.config.use_cache = True
    model.eval()

    try:
        for start in tqdm(range(0, len(sample), batch_size), desc="Year generation eval"):
            batch_examples = sample[start:start + batch_size]

            questions = [ex["user_text"] for ex in batch_examples]
            golds = [ex["final_text"] for ex in batch_examples]

            raw_preds = batch_generate_answers(
                questions,
                max_new_tokens=max_new_tokens,
                temperature=0.0,
                top_p=1.0,
            )

            for ex, gold, raw_pred in zip(batch_examples, golds, raw_preds):
                pred = extract_final(raw_pred)
                s = year_prf(pred, gold)
                scores.append(s)

                examples.append({
                    "question": ex["user_text"],
                    "gold": gold,
                    "pred": pred,
                    "gold_years": sorted(list(extract_year_set(gold))),
                    "pred_years": sorted(list(extract_year_set(pred))),
                    **s,
                })

    finally:
        if old_use_cache is not None:
            model.config.use_cache = old_use_cache

    if not scores:
        return {}, []

    avg = {
        k: float(np.nanmean([x[k] for x in scores]))
        for k in scores[0].keys()
    }

    avg["n_examples"] = len(examples)
    avg["batch_size"] = batch_size
    avg["max_new_tokens"] = max_new_tokens

    return avg, examples


year_metrics, year_examples = evaluate_year_metric_only_gold_has_year_fast(
    dataset_raw["test"],
    n=50,
    batch_size=4,
    max_new_tokens=256,
)

print("YEAR METRICS")
print(year_metrics)

Candidates with year: 4901


Year generation eval:   0%|          | 0/13 [00:00<?, ?it/s]

YEAR METRICS
{'year_precision': 1.0, 'year_recall': 1.0, 'year_f1': 1.0, 'n_examples': 50, 'batch_size': 4, 'max_new_tokens': 256}


In [29]:
# Cell 27 — Inspect bad year examples

if year_examples:
    bad = sorted(year_examples, key=lambda x: x["year_f1"])[:10]

    for ex in bad:
        print("=" * 100)
        print("YEAR F1:", ex["year_f1"])
        print("QUESTION:", ex["question"])
        print("\nGOLD YEARS:", extract_year_set(ex["gold"]))
        print("PRED YEARS:", extract_year_set(ex["pred"]))
        print("\nGOLD:\n", ex["gold"])
        print("\nPRED:\n", ex["pred"])
else:
    print("No year examples found.")

YEAR F1: 1.0
QUESTION: Giải thích vì sao Toàn thắng chống Minh (Tốt Động–Chúc Động, Chi Lăng–Xương Giang) được coi là cột mốc quan trọng.

GOLD YEARS: {'1427'}
PRED YEARS: {'1427'}

GOLD:
 Toàn thắng chống Minh (Tốt Động–Chúc Động, Chi Lăng–Xương Giang) (1427): Nghĩa quân Lam Sơn giành thắng lợi quyết định, buộc quân Minh rút. Sự kiện này có ý nghĩa là Chấm dứt thời Minh thuộc, khôi phục độc lập.

PRED:
 Toàn thắng chống Minh (Tốt Động–Chúc Động, Chi Lăng–Xương Giang) (1427): Nghĩa quân Lam Sơn giành thắng lợi quyết định, buộc quân Minh rút. Sự kiện này có ý nghĩa là Chấm dứt thời Minh thuộc, khôi phục độc lập.
YEAR F1: 1.0
QUESTION: Trong năm 1945, điều gì có ý nghĩa lịch sử ở Việt Nam?

GOLD YEARS: {'1945'}
PRED YEARS: {'1945'}

GOLD:
 Năm 1945, Cách mạng Tháng Tám và Tuyên ngôn Độc lập. Tổng khởi nghĩa giành chính quyền; 2/9/1945, Hồ Chí Minh đọc Tuyên ngôn Độc lập. Về lâu dài, Khai sinh nước Việt Nam Dân chủ Cộng hòa, chấm dứt chế độ thực dân – phong kiến.

PRED:
 **Cách mạng Tháng

In [31]:
# Inspect suspicious dataset rows

def search_raw_examples(keyword, split="train", limit=20):
    found = []
    data = dataset_raw[split]

    for ex in data:
        text = (
            ex.get("user_text", "") + "\n" +
            ex.get("analysis_text", "") + "\n" +
            ex.get("final_text", "")
        ).lower()

        if keyword.lower() in text:
            found.append(ex)
            if len(found) >= limit:
                break

    print(f"Found {len(found)} examples for keyword: {keyword}")

    for i, ex in enumerate(found):
        print("=" * 100)
        print("IDX:", i)
        print("USER:")
        print(ex["user_text"])
        print("\nANALYSIS:")
        print(ex["analysis_text"][:1000])
        print("\nFINAL:")
        print(ex["final_text"][:1000])


search_raw_examples("dời đô", split="train", limit=10)
search_raw_examples("Bình Ngô đại cáo", split="train", limit=10)
search_raw_examples("Lê sơ", split="train", limit=10)

Found 10 examples for keyword: dời đô
IDX: 0
USER:
Những nguyên nhân chính dẫn đến Chiếu dời đô (văn kiện) là gì?

ANALYSIS:
B1: gắn mốc 1010 với Chiếu dời đô (văn kiện). B2: nêu diễn biến trọng tâm – "Văn kiện nêu lý do dời đô và tầm nhìn phát triển ở Đại La.". B3: kết luận ý nghĩa – Tư tưởng quy hoạch, tầm nhìn chiến lược thời Lý..

FINAL:
**Chiếu dời đô (văn kiện)** xảy ra năm 1010. Văn kiện nêu lý do dời đô và tầm nhìn phát triển ở Đại La. Đây là cột mốc quan trọng vì Tư tưởng quy hoạch, tầm nhìn chiến lược thời Lý.
IDX: 1
USER:
Giải thích vì sao Chiếu dời đô – dời đô ra Thăng Long được coi là cột mốc quan trọng.

ANALYSIS:
B1: gắn mốc 1010 với Chiếu dời đô – dời đô ra Thăng Long. B2: nêu diễn biến trọng tâm – "Vua Lý Thái Tổ ban Chiếu dời đô, dời kinh đô từ Hoa Lư ra Đại La (Thăng Long).". B3: kết luận ý nghĩa – Đặt nền móng cho Thăng Long – Hà Nội hơn một nghìn năm văn hiến..

FINAL:
Vào năm 1010, diễn ra **Chiếu dời đô – dời đô ra Thăng Long**: Vua Lý Thái Tổ ban Chiếu dời đô, d

In [ ]:
# Cell 28 — Optional: load saved adapter later after restarting notebook

# from peft import PeftModel

# base_model = AutoModelForCausalLM.from_pretrained(
#     MODEL_ID,
#     quantization_config=bnb_config,
#     device_map="auto",
#     trust_remote_code=True,
#     torch_dtype=compute_dtype,
# )

# model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
# model.eval()

In [30]:
manual_questions = [
    "Việc Lý Công Uẩn dời đô ra Thăng Long năm 1010 có ý nghĩa gì?",
    "Bình Ngô đại cáo ra đời trong bối cảnh nào?",
    "So sánh vai trò của nhà Lý và nhà Trần trong xây dựng và bảo vệ Đại Việt.",
    "Khởi nghĩa Lam Sơn diễn ra trong bối cảnh nào và kết quả ra sao?",
    "Chiến thắng Bạch Đằng năm 938 có ý nghĩa lịch sử gì?",
    "Nhà Trần đã làm gì để chống quân Nguyên Mông?",
    "Cải cách của Hồ Quý Ly có những điểm chính nào?",
    "Vì sao Quang Trung đại phá quân Thanh năm 1789 là sự kiện quan trọng?",
]

model.config.use_cache = True
model.eval()

for q in manual_questions:
    raw = generate_answer(q, max_new_tokens=256, temperature=0.0, top_p=1.0)
    final = extract_final(raw)

    print("=" * 100)
    print("QUESTION:", q)
    print("\nFINAL:")
    print(final)

QUESTION: Việc Lý Công Uẩn dời đô ra Thăng Long năm 1010 có ý nghĩa gì?

FINAL:
Năm 1010, Lý Công Uẩn dời đô ra Thăng Long. Vua Lý Thái Tổ ban Bình Ngô đại cáo, tuyên cáo độc lập. Về lâu dài, Mở ra thời kỳ Lê sơ thịnh trị.
QUESTION: Bình Ngô đại cáo ra đời trong bối cảnh nào?

FINAL:
Năm 1285, Bình Ngô đại cáo. Trần Hưng Đạo soạn Bình Ngô đại cáo tuyên cáo độc lập. Về lâu dài, Khẳng định chủ quyền và cơ cấu quản lý thống nhất.
QUESTION: So sánh vai trò của nhà Lý và nhà Trần trong xây dựng và bảo vệ Đại Việt.

FINAL:
Năm 1009, Lý Công Uẩn được tôn lập (kết thúc Tiền Lê). Sau khi Lê Long Đĩnh qua đời, Lý Công Uẩn được tôn làm vua. Về lâu dài, Mở ra triều Lý với nhiều cải cách và thịnh trị.
QUESTION: Khởi nghĩa Lam Sơn diễn ra trong bối cảnh nào và kết quả ra sao?

FINAL:
Vào năm 1418, diễn ra **Khởi nghĩa Lam Sơn**: Lê Lợi dựng cờ khởi nghĩa ở Lam Sơn chống quân Minh. Ý nghĩa: Khởi đầu cuộc giải phóng dân tộc kéo dài gần 10 năm.
QUESTION: Chiến thắng Bạch Đằng năm 938 có ý nghĩa lịch sử

## Gợi ý chạy thử

1. Chạy với `MAX_SAMPLES = 100_000`, `MAX_LENGTH = 1024` trước.
2. Nếu loss quá thấp bất thường, xem Cell 19 có decode ra đúng `<analysis>` và `<final>` không.
3. Nếu OOM, giảm:

```python
PER_DEVICE_TRAIN_BATCH_SIZE = 2
PER_DEVICE_EVAL_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 8
```

hoặc:

```python
PER_DEVICE_TRAIN_BATCH_SIZE = 1
PER_DEVICE_EVAL_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 16
```

4. Khi mọi thứ ổn, thử `MAX_LENGTH = 2048` hoặc tăng `MAX_SAMPLES`.